# 🎮 GPU-Accelerated Game Analytics: pandas vs `cudf.pandas`

用 **RAPIDS cuDF** 零改碼加速遊戲營運分析（RFM 分群、留存 cohort、漏斗）。

**How to run (Google Colab):**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells twice: 先 `USE_GPU = False` 跑 CPU baseline，再改 `True` 重啟 runtime 跑 GPU
3. 或直接看最下方的 benchmark 結果表

分析邏輯完全是標準 pandas 語法 — `cudf.pandas` 以 zero-code-change 方式攔截加速。

In [ ]:
USE_GPU = True  # 改 False 跑 pandas baseline（需重啟 runtime 才會生效）

if USE_GPU:
    %load_ext cudf.pandas

In [ ]:
import pandas as pd
import numpy as np
import time

print("backend check:", pd)  # cudf.pandas 啟用時會顯示攔截後的模組

## 1. 產生合成遊戲事件資料

模擬一款手遊 90 天、50 萬玩家、約 2,000 萬筆事件（login / stage_clear / purchase / gacha）。
欄位設計參照實際遊戲營運的 event tracking schema。

In [ ]:
N_PLAYERS = 500_000
N_EVENTS = 20_000_000
rng = np.random.default_rng(42)

t0 = time.perf_counter()
player_ids = rng.integers(0, N_PLAYERS, N_EVENTS)
# 玩家活躍度呈冪律：少數重度玩家貢獻大量事件
days = rng.exponential(scale=25, size=N_EVENTS).astype(int) % 90
events = pd.DataFrame({
    "player_id": player_ids,
    "day": days,
    "event_type": rng.choice(
        ["login", "stage_clear", "gacha", "purchase"],
        size=N_EVENTS, p=[0.45, 0.35, 0.15, 0.05]),
    "revenue": np.where(
        rng.random(N_EVENTS) < 0.05,
        rng.lognormal(mean=3.0, sigma=1.0, size=N_EVENTS), 0.0),
})
events["install_day"] = events.groupby("player_id")["day"].transform("min")
print(f"generated {len(events):,} rows in {time.perf_counter()-t0:.1f}s")
events.head()

## 2. RFM 分群

Recency（最近活躍）、Frequency（事件頻次）、Monetary（付費金額）— 遊戲營運最常用的玩家價值分群。

In [ ]:
t0 = time.perf_counter()
rfm = events.groupby("player_id").agg(
    recency=("day", "max"),
    frequency=("event_type", "count"),
    monetary=("revenue", "sum"),
)
rfm["recency"] = 89 - rfm["recency"]
rfm["R"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["segment"] = np.select(
    [
        (rfm.R >= 3) & (rfm.F >= 3) & (rfm.M >= 3),
        (rfm.R >= 3) & (rfm.F >= 3),
        (rfm.R <= 2) & (rfm.M >= 3),
        (rfm.R <= 2) & (rfm.F <= 2),
    ],
    ["whale_active", "core_engaged", "churning_payer", "at_risk"],
    default="casual",
)
t_rfm = time.perf_counter() - t0
print(f"RFM done in {t_rfm:.2f}s")
rfm["segment"].value_counts()

## 3. 留存 Cohort（D1 / D7 / D30）

In [ ]:
t0 = time.perf_counter()
logins = events[events.event_type == "login"][["player_id", "day", "install_day"]].copy()
logins["age"] = logins["day"] - logins["install_day"]
cohort_size = logins.groupby("install_day")["player_id"].nunique()
retention = {}
for d in [1, 7, 30]:
    ret = logins[logins.age == d].groupby("install_day")["player_id"].nunique()
    retention[f"D{d}"] = (ret / cohort_size).fillna(0)
retention_df = pd.DataFrame(retention)
t_ret = time.perf_counter() - t0
print(f"retention done in {t_ret:.2f}s")
retention_df.head(10)

## 4. 付費漏斗（login → stage_clear → gacha → purchase）

In [ ]:
t0 = time.perf_counter()
funnel_sets = {
    step: set(events.loc[events.event_type == step, "player_id"].unique())
    for step in ["login", "stage_clear", "gacha", "purchase"]
}
base = funnel_sets["login"]
funnel = {}
current = base
for step in ["login", "stage_clear", "gacha", "purchase"]:
    current = current & funnel_sets[step]
    funnel[step] = len(current) / max(len(base), 1)
t_funnel = time.perf_counter() - t0
print(f"funnel done in {t_funnel:.2f}s")
funnel

## 5. Benchmark 結果

| 步驟 | pandas (Colab CPU) | cudf.pandas (T4 GPU) | 加速倍數 |
|------|-------------------|----------------------|---------|
| RFM 分群 (20M rows) | *執行後填入* | *執行後填入* | — |
| 留存 cohort | *執行後填入* | *執行後填入* | — |
| 付費漏斗 | *執行後填入* | *執行後填入* | — |

> 兩種模式各跑一次後，把上方各 cell 的計時填入此表。

In [ ]:
print(f"RFM:       {t_rfm:.2f}s")
print(f"Retention: {t_ret:.2f}s")
print(f"Funnel:    {t_funnel:.2f}s")
print("backend:", "cudf.pandas (GPU)" if USE_GPU else "pandas (CPU)")

## 6. 加速比視覺化

跑完 CPU 與 GPU 兩趟後，把兩組計時填入下方 `cpu` / `gpu` dict，執行即產生加速比長條圖。
（範例已預填一組示意值，換成你實跑的數字。）

In [ ]:
import matplotlib.pyplot as plt

# 換成你實跑的秒數
cpu = {"RFM": t_rfm, "Retention": t_ret, "Funnel": t_funnel}
# 第二趟 GPU 跑完後填這裡（或手動貼上）：
gpu = {"RFM": None, "Retention": None, "Funnel": None}

steps = list(cpu.keys())
have_gpu = all(v is not None for v in gpu.values())

fig, axes = plt.subplots(1, 2 if have_gpu else 1, figsize=(11 if have_gpu else 6, 4.2))
axes = axes if have_gpu else [axes]

# 左：時間對比
x = range(len(steps))
w = 0.38
axes[0].bar([i - w/2 for i in x], [cpu[s] for s in steps], w, label="pandas (CPU)", color="#4b5563")
if have_gpu:
    axes[0].bar([i + w/2 for i in x], [gpu[s] for s in steps], w, label="cudf.pandas (GPU)", color="#76b900")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(steps)
axes[0].set_ylabel("seconds"); axes[0].set_title("Execution time per step")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

# 右：加速比
if have_gpu:
    speedup = [cpu[s] / gpu[s] for s in steps]
    bars = axes[1].bar(steps, speedup, color="#76b900")
    axes[1].axhline(1, color="#4b5563", ls="--", lw=1)
    axes[1].set_ylabel("speedup (×)"); axes[1].set_title("GPU speedup vs pandas")
    axes[1].grid(axis="y", alpha=0.3)
    for b, s in zip(bars, speedup):
        axes[1].text(b.get_x() + b.get_width()/2, s, f"{s:.1f}×", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("benchmark_result.png", dpi=120, bbox_inches="tight")
plt.show()
print("saved benchmark_result.png")

## PM Takeaways

- **零改碼**是 cuDF 對數據團隊最大的賣點：不需要重寫既有 pandas pipeline，`%load_ext cudf.pandas` 一行切換。
- 遊戲營運的日常分析（RFM/留存/漏斗）在千萬級事件量時，CPU 上以分鐘計 — GPU 加速讓「每日報表」變成「即時互動查詢」，改變的是營運節奏而不只是省時間。
- 落地評估：Colab T4 免費可驗證；量產可評估 NVIDIA A10/L4 執行個體成本 vs 分析師等待時間。

*Author: Jerome Kuo — gaming data PM (GA/RFM/retention at Gamania & Imperium), NVIDIA DLI certified.*